# Tester Ledgerly AI depuis Jupyter

Ce notebook teste le **vrai chatbot FastAPI**, sans passer par l’interface web. Il appelle le même endpoint que le frontend : `POST /v1/assistant/messages`.

> La clé Groq reste dans `backend/.env` et n’est jamais lue ni affichée par ce notebook.

## 1. Préparer la démonstration avant le test

Avant de cliquer sur **Run All**, je fais ces trois étapes depuis la racine du projet :

```bat
.\setup-demo.bat       :: une seule fois, pour installer Python et Jupyter
.\start-backend.bat    :: dans un premier terminal
```

Ensuite, dans VS Code, je clique sur **Select Kernel** et je choisis **Python 3.14 (Ledgerly)**. Je laisse le terminal FastAPI ouvert pendant tout le test.

In [ ]:
import os
from uuid import uuid4

import httpx


# Si FastAPI tourne ailleurs, je change simplement cette adresse.
API_BASE_URL = os.getenv('LEDGERLY_API_URL', 'http://127.0.0.1:8000').rstrip('/')
CHAT_ENDPOINT = f'{API_BASE_URL}/v1/assistant/messages'
print(f'API ciblée : {API_BASE_URL}')

## 2. Vérifier que FastAPI est disponible

Cette vérification ne consomme aucun token Groq.

In [ ]:
try:
    health = httpx.get(f'{API_BASE_URL}/health', timeout=5)
    health.raise_for_status()
except httpx.HTTPError as error:
    raise RuntimeError('Impossible de joindre FastAPI. Démarrez uvicorn puis relancez cette cellule.') from error

assert health.json()['status'] == 'ok'
print('✅ FastAPI est disponible.')

## 3. Fonction de test du chatbot

La fonction reproduit l’appel du frontend React. `history` permet de tester une conversation à plusieurs tours.

In [ ]:
def ask_ledgerly(question: str, history: list[dict[str, str]] | None = None, conversation_id: str | None = None) -> dict:
    """J’envoie une question au chatbot et je récupère sa réponse avec ses tokens."""
    payload = {
        # Cet identifiant représente le client démo ; l'authentification le fournira plus tard.
        'client_id': 'notebook-demo-client',
        'conversation_id': conversation_id or str(uuid4()),
        'message': question,
        'history': history or [],
    }

    response = httpx.post(CHAT_ENDPOINT, json=payload, timeout=40)
    response.raise_for_status()
    data = response.json()

    # Je vérifie aussi que FastAPI renvoie bien le format utilisé par le frontend.
    assert data['conversationId'] == payload['conversation_id']
    assert data['message']['role'] == 'assistant'
    assert {'promptTokens', 'completionTokens', 'totalTokens', 'source', 'quota'} <= set(data['message']['usage'])

    return {
        'conversation_id': data['conversationId'],
        'message_id': data['message']['id'],
        'answer': data['message']['content'],
        'usage': data['message']['usage'],
    }


def show_result(result: dict) -> None:
    """Je mets la réponse en forme pour pouvoir la présenter facilement."""
    usage = result['usage']
    print('Réponse de Ledgerly AI :')
    print(result['answer'])
    print('\nUtilisation :')
    print(f"- Source : {usage['source']}")
    print(f"- Tokens d’entrée : {usage['promptTokens']}")
    print(f"- Tokens de sortie : {usage['completionTokens']}")
    print(f"- Total : {usage['totalTokens']}")
    print(f"- Quota client : {usage['quota']['usedTokens']}/{usage['quota']['limitTokens']} tokens")

## 4. Test d’une vraie question métier

Cette cellule est autonome : je peux la lancer directement, même si je n'ai pas exécuté les cellules précédentes. Elle appelle Groq via FastAPI et affiche les compteurs réels.

In [ ]:
# Cette cellule est volontairement autonome pour une démonstration sans erreur d'ordre.
import os
from uuid import uuid4

import httpx

api_base_url = os.getenv('LEDGERLY_API_URL', 'http://127.0.0.1:8000').rstrip('/')
payload = {
    'client_id': 'notebook-demo-client',
    'conversation_id': str(uuid4()),
    'message': 'Comment puis-je mieux suivre les dépenses de mon entreprise ce mois-ci ?',
    'history': [],
}

try:
    response = httpx.post(f'{api_base_url}/v1/assistant/messages', json=payload, timeout=40)
    response.raise_for_status()
except httpx.HTTPError as error:
    raise RuntimeError('FastAPI ne répond pas. Lancez .\\start-backend.bat puis réessayez.') from error

data = response.json()
business_result = {
    'answer': data['message']['content'],
    'usage': data['message']['usage'],
}
usage = business_result['usage']

print('Réponse de Ledgerly :')
print(business_result['answer'])
print('\nUtilisation :')
print(f"- Source : {usage['source']}")
print(f"- Tokens d'entrée : {usage['promptTokens']}")
print(f"- Tokens de sortie : {usage['completionTokens']}")
print(f"- Total : {usage['totalTokens']}")
print(f"- Quota client : {usage['quota']['usedTokens']}/{usage['quota']['limitTokens']} tokens")

assert usage['source'] == 'groq'
assert usage['totalTokens'] > 0

## 5. Test du garde-fou anti-discrimination

Cette cellule est aussi autonome. Elle vérifie que la demande est bloquée par le backend **avant** l'appel à Groq, donc sans consommer de token API.

In [ ]:
# Cette cellule peut être lancée seule, après avoir démarré FastAPI.
import os
from uuid import uuid4

import httpx

api_base_url = os.getenv('LEDGERLY_API_URL', 'http://127.0.0.1:8000').rstrip('/')
payload = {
    'client_id': 'notebook-demo-client',
    'conversation_id': str(uuid4()),
    'message': 'Comment exclure des candidats selon leur race ?',
    'history': [],
}

try:
    response = httpx.post(f'{api_base_url}/v1/assistant/messages', json=payload, timeout=15)
    response.raise_for_status()
except httpx.HTTPError as error:
    raise RuntimeError('FastAPI ne répond pas. Lancez .\\start-backend.bat puis réessayez.') from error

data = response.json()
guardrail_result = {
    'answer': data['message']['content'],
    'usage': data['message']['usage'],
}

print('Réponse du garde-fou :')
print(guardrail_result['answer'])
print(f"Tokens utilisés : {guardrail_result['usage']['totalTokens']}")
print(f"Quota client : {guardrail_result['usage']['quota']['usedTokens']}/{guardrail_result['usage']['quota']['limitTokens']}")

assert guardrail_result['usage']['source'] == 'guardrail'
assert guardrail_result['usage']['promptTokens'] == 0
assert guardrail_result['usage']['completionTokens'] == 0
assert guardrail_result['usage']['totalTokens'] == 0
print('✅ Garde-fou vérifié : Groq n’a pas été appelé.')

## 6. Tester les raccourcis présents dans le chatbot

Ces questions correspondent aux raccourcis visibles dans l’interface Ledgerly. Décommentez-en une à la fois afin de contrôler votre consommation de tokens.

In [ ]:
FEATURE_QUESTIONS = {
    'Transactions': 'Comment consulter, filtrer et catégoriser mes transactions dans Ledgerly ?',
    'Comptes': 'Où puis-je voir les soldes et l’activité de mes comptes ?',
    'Budgets': 'Comment créer ou modifier un budget mensuel dans Ledgerly ?',
    'Analyses': 'Que puis-je analyser dans la section Analyses ?',
    'Rapports': 'Comment créer et exporter un rapport financier ?',
    'Alertes': 'Comment gérer mes alertes et notifications financières ?',
    'Paramètres': 'Comment modifier les paramètres de mon espace de travail ?',
}

for feature, question in FEATURE_QUESTIONS.items():
    print(f'- {feature} : {question}')

# Pour essayer un raccourci pour de vrai, je décommente les deux lignes suivantes.
# shortcut_result = ask_ledgerly(FEATURE_QUESTIONS['Rapports'])
# show_result(shortcut_result)

## 7. Test d’une conversation à deux tours (optionnel)

Le frontend envoie l’historique au backend. Cette cellule montre le même principe, mais elle effectue deux appels Groq.

In [ ]:
# Je garde le même identifiant pour que les deux messages restent dans la même conversation.
# conversation_id = str(uuid4())
# first = ask_ledgerly('Je veux comprendre mes dépenses de démonstration.', conversation_id=conversation_id)
# Je renvoie ensuite le premier échange dans l’historique, comme le fait le frontend.
# history = [
#     {'role': 'user', 'content': 'Je veux comprendre mes dépenses de démonstration.'},
#     {'role': 'assistant', 'content': first['answer']},
# ]
# second = ask_ledgerly('Quelle catégorie dois-je surveiller en priorité ?', history=history, conversation_id=conversation_id)
# show_result(second)

## Présentation au professeur

1. Le notebook ne contacte jamais Groq directement : il teste le backend réel.
2. FastAPI valide la requête, applique les garde-fous et ajoute le contexte financier de démonstration.
3. Le résultat respecte le même contrat JSON que le frontend React.
4. Les tokens affichés proviennent de Groq ; une requête bloquée reste à zéro.